In [0]:
"""
Batch Inference Notebook

This notebook performs batch inference using a registered ML model on input data stored in a Delta table.
It is designed to be executed as part of the batch_inference_job workflow defined in
mlops_dbx/resources/batch-inference-workflow-resource.yml.

Parameters:
    env (str, optional): Environment name ("dev", "staging", "prod"). Defaults to "dev".
    input_table_name (str, required): Delta table name containing input features.
    output_table_name (str, required): Delta table name to store output predictions.
    label_table_name (str, optional): Delta table name containing customer labels.
    inference_table_name (str, optional): Delta table name for storing inference results.
    model_name (str, required): Unity Catalog registered model name for batch inference.

Workflow:
    1. Reads notebook parameters using Databricks widgets.
    2. Retrieves the model version using MLflow and a specified alias.
    3. Loads input data from the specified Delta table.
    4. Runs batch prediction using the loaded model.
    5. Joins predictions with customer labels and writes results to the inference table.
    6. Enables Delta Change Data Feed on the inference table.
    7. Exits the notebook, returning the output table name.

Notes:
    - The notebook uses Spark DataFrames for data processing.
    - Predictions are appended to the inference table with a timestamp.
    - The notebook is intended for use in Databricks workflows and jobs.
"""

In [0]:
# List of input args needed to run the notebook as a job.
# Provide them via DB widgets or notebook arguments.
#
# Name of the current environment
dbutils.widgets.dropdown("env", "dev", ["dev", "staging", "prod"], "Environment Name")
# A Hive-registered Delta table containing the input features.
dbutils.widgets.text("input_table_name", "telco_churn_inference_raw", label="Input Table Name")
# Delta table to store the output predictions.
dbutils.widgets.text("output_table_name", "telco_churn_scores", label="Output Table Name")
dbutils.widgets.text("label_table_name", "telco_cust_labels", label="Label Table Name")
dbutils.widgets.text("inference_table_name", "telco_churn_inference_table", label="Inference Table Name")
# Unity Catalog registered model name to use for the trained mode.
dbutils.widgets.text(
    "model_name", "telco_churn_model", label="Full (Three-Level) Model Name"
)
dbutils.widgets.text(
    "username",
    "",
    label="Username",
)
dbutils.widgets.text("catalog_name","mlops_dbx_talk_dev", "Catalog Name")

In [0]:
username = dbutils.widgets.get("username")
if username=="" or username is None:
    raise Exception("Provide the username")
catalog_name = dbutils.widgets.get("catalog_name")

env = dbutils.widgets.get("env")
input_table_name = f"{catalog_name}.{username}.{dbutils.widgets.get('input_table_name')}"
output_table_name = f"{catalog_name}.{username}.{dbutils.widgets.get('output_table_name')}"
label_table_name = f"{catalog_name}.{username}.{dbutils.widgets.get('label_table_name')}"
inference_table_name = f"{catalog_name}.{username}.{dbutils.widgets.get('inference_table_name')}"
model_name = f"{catalog_name}.{username}.{dbutils.widgets.get('model_name')}"
assert input_table_name != "", "input_table_name notebook parameter must be specified"
assert output_table_name != "", "output_table_name notebook parameter must be specified"
assert model_name != "", "model_name notebook parameter must be specified"
alias = "champion"
model_uri = f"models:/{model_name}@{alias}"

In [0]:
from mlflow import MlflowClient

# Get model version from alias
client = MlflowClient(registry_uri="databricks-uc")
model_version = client.get_model_version_by_alias(model_name, alias).version

In [0]:
# Get datetime
from datetime import datetime

ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

In [0]:
import sys
sys.path.append('../..')
from batch_inference.predict import predict_batch
import pyspark.sql.functions as f

input_table = spark.table(input_table_name).select(
                f.col("customerID").alias("customer_id"))

predict_batch(model_uri, input_table, output_table_name, model_version, ts)

df_curr_preds = spark.table(output_table_name).join(spark.table(label_table_name), on='customer_id', how='inner').select('customer_id','model_id',f.col('prediction').cast('integer'),'churn','timestamp')

df_curr_preds.write.mode("append").saveAsTable(inference_table_name)

spark.sql(f"ALTER TABLE {inference_table_name} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")

dbutils.notebook.exit(output_table_name)